# CFP_TimesFM_Forecasts

Generates 1-step-ahead VaR and ES forecasts using Google TimesFM 2.5 (200M). Uses native quantile head for deciles; Student-t tail extrapolation for sub-10% VaR levels. Retains fitted df and computes ES at 2.5% (FRTB).

**Paper:** Pele, Lessmann, Hardle (2026)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path('cfp_ijf_data')
RET_DIR = DATA_DIR / 'returns'

ASSETS = [
    'SP500','STOXX','GDAXI','FCHI','FTSE100','ICLN','NIKKEI','HSI','BOVESPA','NIFTY','ASX200',
    'CBU0','TLT','IBGL','DJCI','GOLD','WTI','NATGAS','BTC','ETH','EURUSD','GBPUSD','USDJPY','AUDUSD'
]
ALPHAS = [0.01, 0.025, 0.05, 0.10]
CONTEXT = 512
N_SAMPLES = 1000

def load_returns(asset):
    df = pd.read_csv(RET_DIR / f'{asset}.csv', parse_dates=['date']).set_index('date').sort_index()
    df = df[df['log_return'].abs() <= 0.50]
    return df['log_return']

print(f'Assets: {len(ASSETS)}, Context: {CONTEXT}, Samples: {N_SAMPLES}')

MODELS = [('timesfm25', 'google/timesfm-2.5-200m-pytorch', 'TimesFM-2.5')]

Assets: 24, Context: 512, Samples: 1000


In [6]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.optimize import minimize
from scipy.stats import t as t_dist

# Patch pentru __init__ înainte de import timesfm
from timesfm.timesfm_2p5 import timesfm_2p5_torch
orig_init = timesfm_2p5_torch.TimesFM_2p5_200M_torch.__init__
def patched_init(self, torch_compile=True, config=None, **kwargs):
    orig_init(self, torch_compile=torch_compile, config=config)
timesfm_2p5_torch.TimesFM_2p5_200M_torch.__init__ = patched_init

import timesfm
from timesfm import ForecastConfig

assert torch.cuda.is_available(), "CUDA not available"
device = torch.device("cuda")
print(f"Device: {device}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Loading TimesFM 2.5...")
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)
tfm.compile(ForecastConfig(
    max_horizon=1,
    max_context=CONTEXT,
    per_core_batch_size=1,
))
print("TimesFM loaded.")

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


def fit_student_t(quantile_vals, quantile_levels):
    """Nelder-Mead Student-t fit on (probs, quantiles) pairs."""
    quantile_vals = np.asarray(quantile_vals, dtype=float)
    quantile_levels = np.asarray(quantile_levels, dtype=float)
    
    def objective(params):
        nu, mu, sigma = params
        if nu <= 2 or sigma <= 0:
            return 1e10
        try:
            predicted = t_dist.ppf(quantile_levels, df=nu, loc=mu, scale=sigma)
            return np.sum((predicted - quantile_vals) ** 2)
        except:
            return 1e10
    
    x0 = [5.0, np.median(quantile_vals), max(np.std(quantile_vals), 1e-6)]
    res = minimize(objective, x0, method="Nelder-Mead",
                   options={"maxiter": 1000, "xatol": 1e-6})
    return res.x


# Sanity check
print("\n--- Sanity check ---")
ret_test = load_returns(ASSETS[0])
context_test = ret_test.values[:CONTEXT].astype(np.float32)

point_test, quantile_test = tfm.forecast(horizon=1, inputs=[context_test])
print(f"point shape: {point_test.shape}")
print(f"quantile shape: {quantile_test.shape}")
print(f"point value: {point_test[0, 0]}")
print(f"quantile values [1:]: {quantile_test[0, 0, 1:]}")
print("--- End sanity check ---\n")


out_dir = DATA_DIR / "timesfm25"
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Output: {out_dir}")

label = "TimesFM-2.5"
pbar = tqdm(ASSETS, desc=label)
for asset in pbar:
    pbar.set_description(f"{label} → {asset}")
    
    ret = load_returns(asset)
    n = len(ret)
    vals = ret.values.astype(np.float32)
    dates = ret.index
    
    records = []
    
    for t_idx in range(CONTEXT, n):
        context = vals[t_idx - CONTEXT:t_idx]
        
        point_fc, quantile_fc = tfm.forecast(horizon=1, inputs=[context])
        
        # quantile_fc shape: (1, 1, num_quantiles)
        # Skip index 0 (probabil median sau p0), folosim 1: pentru cele 9 deciles
        q_vals = np.asarray(quantile_fc[0, 0, 1:])
        
        # Ajustare dacă numărul de quantile diferă
        if len(q_vals) != len(QUANTILE_LEVELS):
            if t_idx == CONTEXT:
                print(f"  Note: got {len(q_vals)} quantiles (after skip), "
                      f"expected {len(QUANTILE_LEVELS)}")
            q_vals = q_vals[:len(QUANTILE_LEVELS)]
        
        if any(np.isnan(q_vals)):
            row = {"date": dates[t_idx]}
            for alpha in ALPHAS:
                row[f"VaR_{alpha}"] = np.nan
            row["mean"] = np.nan
            row["std"] = np.nan
            row["df_student"] = np.nan
            row["ES_student_0.025"] = np.nan
            records.append(row)
            continue
        
        nu, mu, sigma = fit_student_t(q_vals, QUANTILE_LEVELS)
        df_fit = max(nu, 2.01)
        sigma_fit = max(sigma, 1e-8)
        
        row = {"date": dates[t_idx]}
        for alpha in ALPHAS:
            if alpha >= 0.10:
                # VaR la α >= 0.10: interpolare liniară pe quantile grid native
                row[f"VaR_{alpha}"] = -np.interp(alpha, QUANTILE_LEVELS, q_vals)
            else:
                # VaR la α < 0.10: extrapolare via Student-t fitted
                row[f"VaR_{alpha}"] = -t_dist.ppf(alpha, df=df_fit,
                                                   loc=mu, scale=sigma_fit)
        
        row["mean"] = mu
        row["std"] = sigma_fit
        row["df_student"] = df_fit
        
        t_alpha = t_dist.ppf(0.025, df_fit)
        tau_alpha = t_dist.pdf(t_alpha, df_fit)
        row["ES_student_0.025"] = -(
            mu - sigma_fit * tau_alpha
            * (df_fit + t_alpha**2)
            / ((df_fit - 1) * 0.025)
        )
        
        records.append(row)
    
    df_out = pd.DataFrame(records).set_index("date")
    df_out.to_parquet(out_dir / f"{asset}.parquet")

print(f"\n✓ TimesFM-2.5: {len(ASSETS)} assets saved to {out_dir}")

Device: cuda
Loading TimesFM 2.5...
TimesFM loaded.

--- Sanity check ---
point shape: (1, 1)
quantile shape: (1, 1, 10)
point value: -0.0007447764510288835
quantile values [1:]: [-0.01677512 -0.0111202  -0.00721537 -0.00390035 -0.00074478  0.00245416
  0.00578885  0.00975181  0.01561446]
--- End sanity check ---

Output: cfp_ijf_data/timesfm25


TimesFM-2.5 → AUDUSD: 100%|██████████| 24/24 [15:09:48<00:00, 2274.50s/it]  


✓ TimesFM-2.5: 24 assets saved to cfp_ijf_data/timesfm25


In [ ]:
# Verify outputs
from pathlib import Path
for model_slug, _, label in MODELS:
    out_dir = DATA_DIR / model_slug
    files = list(out_dir.glob('*.parquet'))
    print(f'{label}: {len(files)} parquet files')